# Day 5-01｜投籃分析 Pipeline 摘要
> Python 籃球運動資料分析課程  
> 讀取 Day 4 的姿態與球軌跡結果，整理成展示用摘要指標。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 讀取 pose angle 與 ball track 輸出。
- 計算核心摘要指標。
- 輸出 JSON 供成果展示使用。

## 完成產出
- `d5_01_shooting_summary.json` 分析摘要檔。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 尋找 Day 4 結果檔。
2. 計算摘要數值。
3. 寫出成果 JSON。


In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
IN_COLAB = False
DRIVE_MOUNTED = False

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("目前這個 Colab runtime 不支援 google.colab.drive.mount，改用 /content 本機路徑。")

bootstrap_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
    Path("/content/basketball_hackathon/course"),
    Path("/content/basketball_hackathon_course"),
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
]

COURSE_ROOT_HINT = None
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        COURSE_ROOT_HINT = candidate
        break

if COURSE_ROOT_HINT is None and IN_COLAB:
    if DRIVE_MOUNTED:
        COURSE_ROOT_HINT = Path("/content/drive/MyDrive/basketball_hackathon/course")
    else:
        COURSE_ROOT_HINT = Path("/content/basketball_hackathon/course")

    if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
        if COURSE_ROOT_HINT.exists():
            shutil.rmtree(COURSE_ROOT_HINT)
        COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
        cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            REPO_BRANCH,
            REPO_URL,
            str(COURSE_ROOT_HINT),
        ]
        print("$", " ".join(cmd))
        subprocess.run(cmd, check=True)

if COURSE_ROOT_HINT is None or not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，或確認課程 repo 已經存在於目前目錄、/content/basketball_hackathon/course 或 Google Drive。"
    )

if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(COURSE_ROOT_HINT, mount_drive=True)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


In [2]:
import json
import pandas as pd
from src.shooting_utils import estimate_release_frame

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_csv = RESULTS / "d4_03_ball_track.csv"

if not pose_csv.exists():
    raise FileNotFoundError(f"找不到姿態結果：{pose_csv}。請先完成 Day 4-02。")
if not ball_csv.exists():
    raise FileNotFoundError(f"找不到球軌跡結果：{ball_csv}。請先完成 Day 4-03。")

pose_df = pd.read_csv(pose_csv)
ball_df = pd.read_csv(ball_csv)

if pose_df.empty:
    raise RuntimeError("d4_02_pose_angles.csv 是空的，請重新執行 Day 4-02。")
if ball_df.empty:
    raise RuntimeError("d4_03_ball_track.csv 是空的，請重新執行 Day 4-03。")

pose_df.head()


,frame,shoulder_x,shoulder_y,elbow_x,elbow_y,wrist_x,wrist_y,hip_x,hip_y,knee_x,knee_y,ankle_x,ankle_y,elbow_angle,knee_angle,shoulder_angle
0,0,516.265945,328.305688,521.971512,344.745827,546.252556,346.013546,535.612793,377.866087,510.079231,401.218472,502.776489,441.831450,112.128095,142.638886,2.184755
1,5,511.975899,327.987878,522.003555,344.575539,545.774727,343.895030,514.871521,375.355539,505.601044,406.827164,500.720406,443.532872,119.514228,171.160812,27.655831
2,10,511.114502,329.507747,529.225464,345.006495,547.700195,339.032099,503.631516,381.740828,505.350990,414.158993,499.235229,445.836868,121.523936,166.036709,57.597011
3,15,511.692657,330.832093,534.630203,343.161371,552.617416,334.114709,501.172447,381.221724,508.715286,411.759982,499.537849,443.662176,125.041206,150.076683,73.533963
4,20,513.875427,332.679770,537.430611,344.690487,552.396431,334.538090,502.460861,381.551356,513.024826,412.506924,502.407265,443.825984,118.831222,142.429852,76.129572


In [3]:
summary = {
    "pose_frames": int(len(pose_df)),
    "max_elbow_angle": float(pose_df["elbow_angle"].max()),
    "min_elbow_angle": float(pose_df["elbow_angle"].min()),
    "max_knee_angle": float(pose_df["knee_angle"].max()),
    "ball_points": int(len(ball_df)),
    "estimated_release_frame": estimate_release_frame(ball_df),
}
summary


{'pose_frames': 224,
 'max_elbow_angle': 179.2980100366999,
 'min_elbow_angle': 75.29236400605446,
 'max_knee_angle': 178.86062024174376,
 'ball_points': 180,
 'estimated_release_frame': 154}

In [4]:
out_json = RESULTS / "d5_01_shooting_summary.json"
out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", out_json)

saved: H:\Repos\basketball-hackathon-course\assets\results\d5_01_shooting_summary.json


這份 summary 不是正式運動科學評估，只是讓學生知道如何把影像分析轉成可報告的數字。

## 本單元產出檔案

- `assets/results/d5_01_shooting_summary.json`：投籃分析摘要。
